[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eduardd76/multiagent-netops-kit/blob/v0.1.4/labs/solutions/demo_ceos.ipynb)

# Instructor demo -- the SAME agents on real Arista cEOS (containerlab) -- SOLUTION

In [ ]:
# SETUP -- run once. Installs the lab package from GitHub (about 15 s).
# No GitHub access? Run the cells in the APPENDIX at the bottom, then come back and run this cell again.
import os, importlib.util
if not os.environ.get("NETOPS_LOCAL") and importlib.util.find_spec("netops") is None:
    get_ipython().run_line_magic("pip", "install -q git+https://github.com/eduardd76/multiagent-netops-kit@v0.1.4")
from netops.llm import make_llm
from netops import render
llm = make_llm()       # instructor endpoint if NETOPS_BASE_URL/NETOPS_MODEL/NETOPS_API_KEY are set, else MockLLM
print("LLM:", render.llm_label(llm))

**Instructor only** (needs Docker + containerlab + a cEOS image; not for Colab). See `ceos/README.md`.

1. `sudo containerlab deploy -t ceos/topology.clab.yml`  (healthy at deploy)
2. `ceos/break.sh` -- 6 flaps that recover, the silent area change, 12 flaps that don't (~2.5 min)
3. Run the cells. The only line that differs from the student labs is `Network.ceos(...)`.
4. `ceos/reset.sh` between runs.

In [ ]:
import os
from netops.network import Network
from netops.orchestrator import new_incident, run_lab1, run_team, contrast
HOSTS = {"BRANCH-7": "clab.orb.local:8011", "HUB-1": "clab.orb.local:8021",     # eAPI published by containerlab
         "BRANCH-8": "clab.orb.local:8012", "HUB-2": "clab.orb.local:8022"}     # change to your lab host
net = Network.ceos(HOSTS)
print("reachability() =", net.reachability())
print("BRANCH-7 neighbors:", net.ospf_neighbors("BRANCH-7"))
print("areas:", net.ospf_interface("BRANCH-7", "Ethernet1").area, "vs", net.ospf_interface("HUB-1", "Ethernet1").area)

## Lab 1 on real gear -- one agent, no gate

In [ ]:
lab1 = run_lab1(new_incident(llm=llm, network=Network.ceos(HOSTS)))
print(lab1)

## Lab 4 on real gear -- the team, the playbook, and the human gate (a volunteer from the room approves live)

In [ ]:
lab4 = run_team(new_incident(llm=llm, network=Network.ceos(HOSTS)))
print(contrast(lab1, lab4))